# Real-Time Hallucination Detection with Strands Hooks

Based on: [StepShield: When, Not Whether to Intervene on Rogue Agents](https://arxiv.org/abs/2601.22136) (Jan 2026)

## The Problem

Demos 01 and 02 evaluate hallucinations **after** the agent finishes. But in production, you want to catch hallucinations **during** execution, before the response reaches the end user.

## The Technique: Hook-Based Grounding Check

Strands Agents provides a hook system that intercepts every tool call and model response. We use this to:

1. **Collect tool outputs** as they happen (`AfterToolCallEvent`)
2. **Check the final response** against collected tool outputs (`AfterModelCallEvent`)
3. **Flag hallucinations** in real-time before they reach the user

```
Agent receives query
  → Calls tool (hook captures output as "ground truth")
  → Calls tool (hook captures output)
  → Generates response (hook checks: is response grounded in tool outputs?)
  → ✅ Grounded → deliver to user
  → ⚠️ Hallucinated → flag before delivery
```

This is the StepShield pattern: evaluate at each step, not after the trajectory completes.

In [ ]:
# %pip install strands-agents strands-agents-evals boto3

## Step 1: Understand the HallucinationDetector hook

**What this does:** Creates an agent with a `HallucinationDetector` hook that automatically checks every response for grounding against tool outputs.

**Why we use the HookProvider pattern:** The `HookProvider` is a protocol (interface) that lets you plug custom logic into the agent's execution lifecycle without modifying the agent itself. You implement `register_hooks()` to subscribe to specific events. The hook system calls your code automatically at the right moments — you never call the hook manually.

The `HallucinationDetector` registers 3 callbacks:

| Event | When It Fires | What the Hook Does |
|-------|--------------|-------------------|
| `BeforeInvocationEvent` | Start of each `agent()` call | Resets collected tool outputs for a fresh check |
| `AfterToolCallEvent` | After every tool call completes | Stores the tool output as ground truth context |
| `AfterModelCallEvent` | After the model generates text | Checks if the response is grounded in collected tool outputs |

The key insight: **tool outputs are the source of truth**. If the agent says something that no tool returned, it is a hallucination. The hook collects tool outputs as they happen and uses them as the reference context for grounding checks.

> **What to look for:** The agent is created with `hooks=[detector]`. The threshold is 0.5 — scores below this flag hallucination. Two tools are available: `search_flights` and `get_weather`.

In [ ]:
import nest_asyncio
nest_asyncio.apply()  # Fix for Jupyter async event loop

from hallucination_hook import HallucinationDetector, search_flights, get_weather
from strands import Agent
from strands.models.openai import OpenAIModel

MODEL = OpenAIModel(model_id="gpt-4o-mini")

# Create the detector hook
detector = HallucinationDetector(model=MODEL, threshold=0.5)

# Attach it to the agent — the hook intercepts every tool call and model response
agent = Agent(
    model=MODEL,
    tools=[search_flights, get_weather],
    hooks=[detector],
    system_prompt="You are a travel assistant. Use tools to answer questions. Only state facts from tool results.",
)

print("✅ Agent created with HallucinationDetector hook attached")
print(f"   Threshold: {detector.threshold} (scores below this flag hallucination)")
print(f"   Tools: search_flights, get_weather")

## Step 2: Run queries and watch the hook in action

**What this does:** Sends two different queries to the agent. The hook runs automatically on every agent call — no manual invocation needed.

**Why two different queries:** Query 1 uses `search_flights` and Query 2 uses `get_weather`. This tests whether the hook correctly collects tool-specific outputs and checks the response against the right context for each query.

> **What to look for in Query 1:** The agent should call `search_flights`, and the hook should check whether the response only mentions flights that the tool actually returned. Look for a grounding score near 1.0 if the agent sticks to the facts, or a warning if it embellishes (adds airlines, prices, or details not in the tool output).

> **What to look for in Query 2:** The agent should call `get_weather`. Watch for the same pattern — the hook checks the response against what `get_weather` returned. Any mention of weather details not in the tool output should lower the grounding score.

In [ ]:
# Query 1: Agent should call search_flights and report grounded results
print("=" * 60)
print("QUERY 1: Find flights from NYC to London for Friday")
print("=" * 60)

result1 = agent("Find flights from NYC to London for Friday")
print(f"\nAgent response:\n{result1}")

In [ ]:
# Query 2: Agent should call get_weather and report grounded results
print("=" * 60)
print("QUERY 2: What's the weather in Paris?")
print("=" * 60)

result2 = agent("What's the weather in Paris?")
print(f"\nAgent response:\n{result2}")

## Step 3: Review all detection results

**What this does:** Displays the complete log of grounding checks performed by the hook across all queries.

**Why we review the summary:** Individual query outputs show real-time detection, but the summary reveals patterns — for example, whether the agent consistently hallucinates on certain tool types, or whether grounding scores are consistently high.

> **What to look for:** Each check shows the grounding score, which tool outputs were used as context, and a preview of the response. Ideally, all responses should be grounded (score >= 0.5). If any are flagged, compare the response preview to the tool sources to understand what was fabricated.

In [ ]:
print("=" * 60)
print("DETECTION SUMMARY")
print("=" * 60)

for i, check in enumerate(detector.checks):
    icon = "✅" if check["is_grounded"] else "⚠️"
    print(f"\n  {icon} Check {i+1}:")
    print(f"     Grounding score: {check['grounding_score']:.2f}")
    print(f"     Tool sources used: {check['context_sources']}")
    print(f"     Response preview: {check['response_preview']}...")

grounded_count = sum(1 for c in detector.checks if c["is_grounded"])
total = len(detector.checks)
print(f"\n📊 Results: {grounded_count}/{total} responses grounded")
print(f"🚨 Hallucinations flagged: {total - grounded_count}")

## How the Hook Works (Code Walkthrough)

The `HallucinationDetector` implements the `HookProvider` protocol:

```python
class HallucinationDetector(HookProvider):
    def register_hooks(self, registry, **kwargs):
        registry.add_callback(BeforeInvocationEvent, self._reset)
        registry.add_callback(AfterToolCallEvent, self._collect_tool_output)
        registry.add_callback(AfterModelCallEvent, self._check_grounding)
```

**`_collect_tool_output`**: After every tool call, extracts the text content and stores it:
```python
def _collect_tool_output(self, event):
    tool_name = event.tool_use["name"]
    for content in event.result.get("content", []):
        if "text" in content:
            self.tool_outputs.append(f"[{tool_name}]: {content['text']}")
```

**`_check_grounding`**: After the model's final response, runs an `OutputEvaluator` to check if the response is grounded in the collected tool outputs:
```python
def _check_grounding(self, event):
    if event.stop_response.stop_reason != "end_turn":
        return  # Only check final responses, not mid-reasoning
    
    context = "\n".join(self.tool_outputs)
    # Run grounding check with OutputEvaluator...
```

The hook runs automatically on every `agent()` call. No manual intervention needed.

## Key Takeaways

1. **Hooks make evaluation part of the agent runtime.** No separate evaluation step needed. The check happens inline during execution.

2. **Tool outputs are the ground truth.** If the agent calls `search_flights` and gets 2 results, any mention of a 3rd flight is a hallucination.

3. **Cost: 1 extra LLM call per agent response.** The grounding check runs after each final response. For latency-sensitive applications, you can run it asynchronously and flag hallucinations after delivery.

4. **This pattern extends to safety.** The same hook architecture can check for toxicity, PII leakage, or policy violations in real-time. See [StepShield](https://arxiv.org/abs/2601.22136).

### Comparison: Post-hoc vs Real-time Detection

| Approach | When | Cost | Latency | Best For |
|----------|------|------|---------|----------|
| OutputEvaluator (Demo 01) | After execution | 1 call | None (async) | Batch evaluation, CI/CD |
| Claim decomposition (Demo 02) | After execution | 1+N calls | None (async) | Root cause analysis |
| RAGAS (Demo 01) | After execution | 1-2 calls | None (async) | RAG pipeline evaluation |
| **Hooks (this demo)** | **During execution** | **1 call** | **Adds latency** | **Production guardrails** |

**Series complete!** You now have 4 approaches to hallucination detection, each suited to different scenarios.